In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import pickle
import os

In [2]:
import os
import pickle
import pandas as pd

df_model = pd.read_csv('processed/df_model.csv')
df_orig = pd.read_csv('processed/classification_with_orf.tsv', sep='\t')

with open('processed/sequences.pkl', 'rb') as f:
    seqs = pickle.load(f)

df = df_orig[['isoform']].copy()
df['sequence'] = df['isoform'].map(seqs)
df = pd.concat([df, df_model], axis=1)

df = df[df['target_4class'].isin(['real_high', 'artifact'])].copy()
df['label'] = (df['target_4class'] == 'real_high').astype(int)

df = df.drop(columns=['target_4class', 'target_3class'])

df = df.dropna(subset=['sequence'])

for col in df.columns:
    if col in ['isoform', 'sequence', 'label']:
        continue
    if df[col].dtype == object:
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

print(f"Размер: {len(df):,}")
print(f"Баланс: {df['label'].value_counts()}")
print(f"Пропуски: {df.isnull().sum().sum()}")
print(f"Колонки: {df.columns.tolist()}")

Размер: 269,326
Баланс: label
1    202421
0     66905
Name: count, dtype: int64
Пропуски: 0
Колонки: ['isoform', 'sequence', 'strand', 'length', 'exons', 'ref_length', 'ref_exons', 'diff_to_gene_TSS', 'diff_to_gene_TTS', 'RTS_stage', 'all_canonical', 'bite', 'perc_A_downstream_TTS', 'protein_length', 'has_orf', 'FSM_class_A', 'FSM_class_B', 'FSM_class_C', 'orf_type_3prime_partial', 'orf_type_5prime_partial', 'orf_type_complete', 'orf_type_internal', 'orf_type_no_orf', 'label']


In [3]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val, test = train_test_split(temp, test_size=0.333, random_state=42, stratify=temp['label'])

train.to_parquet('processed/train.parquet', index=False)
val.to_parquet('processed/val.parquet', index=False)
test.to_parquet('processed/test.parquet', index=False)

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

VOCAB = {'A': 1, 'T': 2, 'G': 3, 'C': 4, 'N': 5}
PAD_IDX = 0

TAB_FEATURES = [
    'strand', 'length', 'exons', 'ref_length', 'ref_exons',
    'diff_to_gene_TSS', 'diff_to_gene_TTS', 'RTS_stage', 'all_canonical',
    'bite', 'perc_A_downstream_TTS', 'protein_length', 'has_orf',
    'FSM_class_A', 'FSM_class_B', 'FSM_class_C',
    'orf_type_3prime_partial', 'orf_type_5prime_partial', 'orf_type_complete',
    'orf_type_internal', 'orf_type_no_orf'
]

def encode_sequence(seq, max_len):
    encoded = [VOCAB.get(c, 5) for c in seq[:max_len]]
    padded = encoded + [PAD_IDX] * (max_len - len(encoded))
    return torch.tensor(padded, dtype=torch.long)

class IsoformDataset(Dataset):
    def __init__(self, df, max_len=2000):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        seq = encode_sequence(row['sequence'], self.max_len)
        tab = torch.tensor(row[TAB_FEATURES].values.astype(np.float32))
        label = torch.tensor(row['label'], dtype=torch.float32)
        
        return seq, tab, label


def get_loaders(batch_size=16, max_len=8000):
    train_df = pd.read_parquet('processed/train.parquet')
    val_df   = pd.read_parquet('processed/val.parquet')
    test_df  = pd.read_parquet('processed/test.parquet')
    
    train_df = train_df[train_df['sequence'].str.len() <= max_len]
    val_df   = val_df[val_df['sequence'].str.len() <= max_len]
    test_df  = test_df[test_df['sequence'].str.len() <= max_len]
    
    print(f"Train: {len(train_df):,}")
    print(f"Val:   {len(val_df):,}")
    print(f"Test:  {len(test_df):,}")
    
    train_loader = DataLoader(IsoformDataset(train_df, max_len), batch_size=batch_size, shuffle=True,  num_workers=4)
    val_loader   = DataLoader(IsoformDataset(val_df,   max_len), batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader  = DataLoader(IsoformDataset(test_df,  max_len), batch_size=batch_size, shuffle=False, num_workers=4)
    
    return train_loader, val_loader, test_loader

In [7]:
def encode_sequence(seq, max_len):
    mapping = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
    encoded = [mapping[c] if c in mapping else -1 for c in seq[:max_len]]
    padded = encoded + [-1] * (max_len - len(encoded))
    return torch.tensor(padded, dtype=torch.long)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2, kernel_size=7):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(2)
        )

    def forward(self, x):
        return self.block(x)

class SeqEncoder(nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.convs = nn.Sequential(
            ConvBlock(4, 128, kernel_size=7, dropout=dropout),
            ConvBlock(128, 256, kernel_size=5, dropout=dropout),
            ConvBlock(256, hidden_dim, kernel_size=3, dropout=dropout),
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        mask = (x == -1)
        one_hot = torch.nn.functional.one_hot(x.clamp(min=0), num_classes=4).float()
        one_hot[mask] = 0
        x = one_hot.permute(0, 2, 1)
        x = self.convs(x)
        x = self.global_pool(x)
        return x.squeeze(-1)

class TabEncoder(nn.Module):
    def __init__(self, input, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.mlp(x)

class IsoformClassifier(nn.Module):
    def __init__(self, tab_input, tab_dim, conv_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.tab = TabEncoder(input=tab_input, hidden_dim=tab_dim, dropout=dropout)
        self.seq = SeqEncoder(conv_dim, dropout=dropout)
        self.mlp = nn.Sequential(
            nn.Linear(conv_dim + tab_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_conv, x_tab):
        x_conv = self.seq(x_conv)
        x_tab = self.tab(x_tab)
        comb = torch.cat([x_conv, x_tab], dim=1)
        return self.mlp(comb).squeeze(-1)


In [9]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, f1_score
from tqdm import tqdm
import numpy as np

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_labels, all_preds = [], []
    
    pbar = tqdm(loader, desc='Train', leave=False)
    for seq, tab, label in pbar:
        seq, tab, label = seq.to(device), tab.to(device), label.to(device)
        
        optimizer.zero_grad()
        logits = model(seq, tab)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        all_labels.extend(label.cpu().numpy())
        all_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    auc = roc_auc_score(all_labels, all_preds)
    return total_loss / len(loader), auc


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_labels, all_preds = [], []
    
    pbar = tqdm(loader, desc='Val', leave=False)
    for seq, tab, label in pbar:
        seq, tab, label = seq.to(device), tab.to(device), label.to(device)
        
        logits = model(seq, tab)
        loss = criterion(logits, label)
        
        total_loss += loss.item()
        all_labels.extend(label.cpu().numpy())
        all_preds.extend(torch.sigmoid(logits).cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    auc = roc_auc_score(all_labels, all_preds)
    f1  = f1_score(all_labels, [p > 0.5 for p in all_preds])
    return total_loss / len(loader), auc, f1


def train(model, train_loader, val_loader, n_epochs=20, lr=1e-3, device='cuda'):
    model = model.to(device)
    
    pos_weight = torch.tensor([66905 / 202421]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)
    
    best_auc = 0
    best_epoch = 0
    
    for epoch in range(n_epochs):
        train_loss, train_auc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_auc, val_f1 = eval_epoch(model, val_loader, criterion, device)
        
        scheduler.step(val_auc)
        
        print(f"Epoch {epoch+1:02d} | "
            f"train loss: {train_loss:.4f} auc: {train_auc:.4f} | "
            f"val loss: {val_loss:.4f} auc: {val_auc:.4f} f1: {val_f1:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'best_model.pt')
            print(f"best model saved")
    
    print(f"\nBest model: epoch {best_epoch}, val AUC: {best_auc:.4f}")


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

model = IsoformClassifier(tab_input=21, tab_dim=64, conv_dim=128, hidden_dim=128)
train_loader, val_loader, test_loader = get_loaders(batch_size=32, max_len=8000)
train(model, train_loader, val_loader, n_epochs=20, lr=1e-3, device=device)

Device: cuda
Train: 185,074
Val:   52,867
Test:  26,425


Epoch 01 | train loss: 0.1053 auc: 0.9719 | val loss: 0.0949 auc: 0.9849 f1: 0.9626
best model saved


Epoch 02 | train loss: 0.0860 auc: 0.9807 | val loss: 0.0731 auc: 0.9881 f1: 0.9628
best model saved


KeyboardInterrupt: 